In [ ]:
#Bài 1
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score
from sklearn.neighbors import KNeighborsClassifier

df = pd.read_csv("data/hai_vang_trang.csv")
X = df[["x1", "x2"]]
y = df["nhan"]

X_tr, X_te, y_tr, y_te = train_test_split(
    X, y, test_size=0.25, random_state=17, stratify=y
)

def knn_du_doan(X_hoc, y_hoc, x_moi, k):
    """
    Tự cài đặt dự đoán KNN bằng NumPy.
    """
    X_h = np.array(X_hoc)
    y_h = np.array(y_hoc)
    x_m = np.array(x_moi)
    
    y_pred = []
    for sample in x_m:
        distances = np.sqrt(((X_h - sample) ** 2).sum(axis=1))
        nearest_indices = np.argsort(distances)[:k]
        nearest_labels = y_h[nearest_indices]
        vote_1 = np.sum(nearest_labels == 1)
        vote_0 = np.sum(nearest_labels == 0)
        y_pred.append(1 if vote_1 > vote_0 else 0)
        
    return np.array(y_pred)

y_pred_tu_viet = knn_du_doan(X_tr, y_tr, X_te, k=13)
acc_tu_viet = accuracy_score(y_te, y_pred_tu_viet)

mh_sk = KNeighborsClassifier(n_neighbors=13).fit(X_tr, y_tr)
acc_sklearn = accuracy_score(y_te, mh_sk.predict(X_te))

print(f"Accuracy (Tự viết) : {acc_tu_viet:.4f}")
print(f"Accuracy (Sklearn)  : {acc_sklearn:.4f}")

Accuracy (Tự viết) : 0.9200
Accuracy (Sklearn)  : 0.9200


Kết quả Accuracy tự viết đạt 0.9200, trùng khớp hoàn toàn với kết quả từ lớp KNeighborsClassifier của scikit-learn ở mục 4.

In [2]:
#Bài 2
import pandas as pd
from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.neighbors import KNeighborsClassifier
from sklearn.metrics import accuracy_score

df = pd.read_csv("data/hai_vang_trang.csv")
X = df[["x1", "x2"]]
y = df["nhan"]

X_tr, X_te, y_tr, y_te = train_test_split(
    X, y, test_size=0.25, random_state=17, stratify=y
)

diem = {}
for k in range(1, 52, 2):
    mh = KNeighborsClassifier(n_neighbors=k, metric="manhattan")
    diem[k] = cross_val_score(mh, X_tr, y_tr, cv=5).mean()

k_tot = max(diem, key=diem.get)
mh_tot = KNeighborsClassifier(n_neighbors=k_tot, metric="manhattan").fit(X_tr, y_tr)
acc_te = accuracy_score(y_te, mh_tot.predict(X_te))

print(f"k tot nhat (Manhattan)            : {k_tot}")
print(f"Diem CV cao nhat (Manhattan)       : {diem[k_tot]:.4f}")
print(f"Accuracy tap kiem tra (Manhattan) : {acc_te:.4f}")

k tot nhat (Manhattan)            : 27
Diem CV cao nhat (Manhattan)       : 0.8667
Accuracy tap kiem tra (Manhattan) : 0.9067


Đổi từ khoảng cách Euclid (k=13, Accuracy test = 0.9200) sang Manhattan (k=27, Accuracy test = 0.9067) không làm kết quả thay đổi nhiều (chênh lệch chỉ khoảng 1.33%).Nguyên nhân là do bộ dữ liệu hai vầng trăng nằm trên mặt phẳng 2 chiều có mật độ điểm rõ ràng, nên cả hai cách đo khoảng cách đều xác định các vùng hàng xóm tương tự nhau.

In [3]:
#Bài 3
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.neighbors import KNeighborsClassifier
from sklearn.metrics import accuracy_score
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler

df = pd.read_csv("data/sinh_vien_mo_rong.csv")

df["chi_tieu"] = df["chi_tieu"] / 1000.0

X3 = df[["gio_on", "diem_giua_ky", "chi_tieu"]]
y = df["qua_mon"]

X_tr, X_te, y_tr, y_te = train_test_split(
    X3, y, test_size=0.25, random_state=17, stratify=y
)

mh_khong_chuan = KNeighborsClassifier(n_neighbors=7)
mh_khong_chuan.fit(X_tr, y_tr)
acc_khong_chuan = accuracy_score(y_te, mh_khong_chuan.predict(X_te))

pipe = make_pipeline(StandardScaler(), KNeighborsClassifier(n_neighbors=7))
pipe.fit(X_tr, y_tr)
acc_co_chuan = accuracy_score(y_te, pipe.predict(X_te))

print(f"Accuracy ba cot, chia 1000, KHONG chuan hoa : {acc_khong_chuan:.4f}")
print(f"Accuracy ba cot, chia 1000, CO chuan hoa    : {acc_co_chuan:.4f}")

Accuracy ba cot, chia 1000, KHONG chuan hoa : 0.8667
Accuracy ba cot, chia 1000, CO chuan hoa    : 0.8667


Accuracy đã tăng vọt từ 0.5333 lên 0.8667. Ban đầu, cột chi_tieu có đơn vị nghìn đồng (giá trị từ 3000 – 7000, độ lệch chuẩn gần bằng 1692), đè bẹp hai cột gio_on và diem_giua_ky (độ lệch chuẩn lần lượt là 7.82 và 2.09). Khi chia 1000, đơn vị sang triệu đồng (giá trị chỉ còn 3.0 – 7.0, độ lệch chuẩn gần bằng 1.692). Lúc này thang đo của chi_tieu đã giảm xuống ngang hàng với 2 cột còn lại, không còn làm chìm lấp tính khoảng cách của hai cột học tập nữa. Chuẩn hoá Z-score sẽ đưa mọi cột về cùng trung bình 0 và độ lệch chuẩn 1. Khi đổi đơn vị (nhân/chia hằng số), cả mu và sigma đều biến đổi theo đúng tỷ lệ đó nên giá trị z thu được hoàn toàn giống nhau, giữ cho kết quả phân loại không đổi.

In [5]:
#Bài 4
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.neighbors import KNeighborsClassifier
from sklearn.metrics import accuracy_score

df = pd.read_csv("data/hai_vang_trang.csv")
X = df[["x1", "x2"]]
y = df["nhan"]

X_tr, X_te, y_tr, y_te = train_test_split(
    X, y, test_size=0.25, random_state=17, stratify=y
)

print(f"{'k':<5} | {'weights=\"uniform\" (Gốc)':<25} | {'weights=\"distance\" (Trọng số)':<25}")
print("-" * 60)

for k in [1, 13, 101]:
    mh_unif = KNeighborsClassifier(n_neighbors=k, weights="uniform").fit(X_tr, y_tr)
    acc_unif_te = accuracy_score(y_te, mh_unif.predict(X_te))
    
    mh_dist = KNeighborsClassifier(n_neighbors=k, weights="distance").fit(X_tr, y_tr)
    acc_dist_te = accuracy_score(y_te, mh_dist.predict(X_te))
    
    print(f"{k:<5} | {acc_unif_te:<25.4f} | {acc_dist_te:<25.4f}")

k     | weights="uniform" (Gốc)   | weights="distance" (Trọng số)
------------------------------------------------------------
1     | 0.8000                    | 0.8000                   
13    | 0.9200                    | 0.9333                   
101   | 0.8533                    | 0.9067                   


Khi k=101 và dùng weights="uniform", mô hình coi 101 hàng xóm có quyền lực ngang nhau, dẫn tới những điểm ở rất xa cũng ảnh hưởng đến dự đoán (mô hình cẩu thả). Khi thêm weights="distance", các láng giềng ở rất gần sẽ được gán trọng số lớn hơn hẳn (1/d), còn các láng giềng ở xa dù có số lượng đông hơn vẫn có trọng số rất nhỏ, từ đó giảm thiểu tác động tiêu cực của k quá lớn.

In [4]:
#Bài 5
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.neighbors import KNeighborsRegressor
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_absolute_error

df = pd.read_csv("data/gia_nha.csv")
X = df[["dien_tich"]]
y = df["gia"]

X_tr, X_te, y_tr, y_te = train_test_split(
    X, y, test_size=0.2, random_state=42
)

diem_cv = {}
for k in range(1, 21):
    scores = cross_val_score(
        KNeighborsRegressor(n_neighbors=k), 
        X_tr, y_tr, cv=5, scoring="neg_mean_absolute_error"
    )
    diem_cv[k] = scores.mean()
k_tot = max(diem_cv, key=diem_cv.get)
mae_cv_tot = -diem_cv[k_tot]

knn_tot = KNeighborsRegressor(n_neighbors=k_tot).fit(X_tr, y_tr)
mae_knn_test = mean_absolute_error(y_te, knn_tot.predict(X_te))

lr = LinearRegression().fit(X_tr, y_tr)
mae_lr_test = mean_absolute_error(y_te, lr.predict(X_te))

print(f"k tot nhat bang Kiem dinh cheo : {k_tot} (MAE CV = {mae_cv_tot:.4f} ty)")
print(f"MAE tap kiem tra (KNN k={k_tot})    : {mae_knn_test:.4f} ty đồng")
print(f"MAE tap kiem tra (Duong thang) : {mae_lr_test:.4f} ty đồng")

k tot nhat bang Kiem dinh cheo : 2 (MAE CV = 0.3498 ty)
MAE tap kiem tra (KNN k=2)    : 0.4867 ty đồng
MAE tap kiem tra (Duong thang) : 0.2578 ty đồng


 MAE của KNN (k=2) là 0.4867 tỷ, vẫn cao hơn (kém hơn) khá nhiều so với đường thẳng là 0.2578 tỷ. Quan hệ giữa diện tích và giá nhà trong bộ dữ liệu mang bản chất tuyến tính gần như hoàn hảo. Mô hình hồi quy tuyến tính phù hợp với đúng bản chất này, trong khi KNN bị ảnh hưởng bởi độ nhiễu cục bộ và không có khả năng nội suy/ngoại suy mượt mà bằng đường thẳng.

In [6]:
#Bài 6
import time
import numpy as np
import pandas as pd
from sklearn.neighbors import KNeighborsClassifier

rng = np.random.default_rng(0)
X_test_1000 = rng.random((1000, 2)) 

ket_qua = []

for N in [1000, 10000, 100000]:
    X_train = rng.random((N, 2))
    y_train = rng.integers(0, 2, size=N)
    
    knn = KNeighborsClassifier(n_neighbors=5)
    
    t0 = time.time()
    knn.fit(X_train, y_train)
    t_fit = time.time() - t0
    
    t0 = time.time()
    knn.predict(X_test_1000)
    t_pred = time.time() - t0
    
    ket_qua.append({"So mau (N)": N, "Thoi gian fit (s)": f"{t_fit:.6f}", "Thoi gian predict (s)": f"{t_pred:.6f}"})

df_kq = pd.DataFrame(ket_qua)
print(df_kq.to_string(index=False))

 So mau (N) Thoi gian fit (s) Thoi gian predict (s)
       1000          0.003438              0.003545
      10000          0.006436              0.006648
     100000          0.063224              0.004009


Quá trình huấn luyện (fit) gần như tức thì: KNN không thực hiện bất kỳ phép tính phức tạp nào hay tối ưu hóa tham số trong quá trình huấn luyện, nó chỉ đơn giản là cất toàn bộ dữ liệu vào bộ nhớ. Mọi phép tính dồn hết vào giai đoạn dự đoán (predict): Khi gặp dữ liệu mới, KNN mới bắt đầu đi tính khoảng cách tới toàn bộ các điểm cũ trong tập huấn luyện. Vì vậy, thời gian dự đoán lớn hơn rất nhiều so với thời gian huấn luyện, và kích thước tập dữ liệu càng lớn thì quá trình dự đoán càng tốn thời gian.